In [ ]:
import numpy as np
import pandas as pd
import json
from pathlib import Path
from datetime import datetime, timedelta

# ─────────────────────────────────────────────
# Config
# ─────────────────────────────────────────────
LOCALITIES_CSV     = "localities_delhi_geocoded.csv"
OUT_INCIDENTS      = "synthetic_wsi_delhi_localities.csv"
OUT_LOCALITY_AGG   = "locality_aggregates.json"
OUT_FEATURE_ORDER  = "feature_order.json"

N_ROWS       = 1_000_000
RANDOM_SEED  = 42

START_DATE = "2023-01-01T00:00:00"
END_DATE   = "2024-12-31T23:59:59"

SPREAD_DEG       = 0.006
P_OFF_LOCALITY   = 0.02

LAT_MIN, LAT_MAX = 28.40, 28.90
LON_MIN, LON_MAX = 76.90, 77.40

np.random.seed(RANDOM_SEED)

# ─────────────────────────────────────────────
# Location type lux profiles
# ─────────────────────────────────────────────
LOCATION_TYPE_PROFILES = {
    "commercial": {
        "expected_night_lux_range": (200, 500),
        "alert_threshold": 50,
        "shadow_lux_threshold": 20,
        "shadow_penalty_value": 0.40,
    },
    "campus": {
        "expected_night_lux_range": (50, 150),
        "alert_threshold": 20,
        "shadow_lux_threshold": 10,
        "shadow_penalty_value": 0.25,
    },
    "highway": {
        "expected_night_lux_range": (30, 50),
        "alert_threshold": 10,
        "shadow_lux_threshold": 5,
        "shadow_penalty_value": 0.20,
    },
    "residential": {
        "expected_night_lux_range": (10, 20),
        "alert_threshold": 5,
        "shadow_lux_threshold": 3,
        "shadow_penalty_value": 0.0,   # baseline — no extra shadow penalty
    },
    "safe_ip": {
        "expected_night_lux_range": (100, 300),
        "alert_threshold": 30,
        "shadow_lux_threshold": 15,
        "shadow_penalty_value": 0.35,
    },
}

# ─────────────────────────────────────────────
# Load localities (now with location_type)
# ─────────────────────────────────────────────
loc_df = pd.read_csv(LOCALITIES_CSV)

required_cols = {"name", "latitude", "longitude", "location_type"}
if not required_cols.issubset(set(loc_df.columns)):
    raise ValueError(f"Localities CSV must contain columns: {', '.join(required_cols)}")

loc_df = loc_df.dropna(subset=["latitude", "longitude"]).reset_index(drop=True)
if loc_df.empty:
    raise ValueError("No geocoded localities found.")

# Fill missing location_type with 'residential'
loc_df["location_type"] = loc_df["location_type"].fillna("residential")
n_loc = len(loc_df)

print(f"Loaded {n_loc} localities")
print(f"Location types: {loc_df['location_type'].value_counts().to_dict()}")

# ─────────────────────────────────────────────
# Sample locality indices + off-locality mask
# ─────────────────────────────────────────────
local_indices = np.random.randint(0, n_loc, size=N_ROWS)
is_off = np.random.rand(N_ROWS) < P_OFF_LOCALITY

# Pre-allocate arrays
lat = np.empty(N_ROWS, dtype=float)
lon = np.empty(N_ROWS, dtype=float)
locality_assigned = np.empty(N_ROWS, dtype=object)
location_type_assigned = np.empty(N_ROWS, dtype=object)

for i in range(N_ROWS):
    if is_off[i]:
        lat[i] = np.random.uniform(LAT_MIN, LAT_MAX)
        lon[i] = np.random.uniform(LON_MIN, LON_MAX)
        locality_assigned[i] = "UNK"
        location_type_assigned[i] = "residential"  # default for unknown
    else:
        li = local_indices[i]
        center_lat = loc_df.loc[li, "latitude"]
        center_lon = loc_df.loc[li, "longitude"]
        lat[i] = center_lat + np.random.normal(0, SPREAD_DEG)
        lon[i] = center_lon + np.random.normal(0, SPREAD_DEG)
        locality_assigned[i] = loc_df.loc[li, "name"]
        location_type_assigned[i] = loc_df.loc[li, "location_type"]

# ─────────────────────────────────────────────
# Timestamps
# ─────────────────────────────────────────────
start_ts = pd.to_datetime(START_DATE)
end_ts   = pd.to_datetime(END_DATE)
total_seconds = int((end_ts - start_ts).total_seconds())
rand_seconds = np.random.randint(0, total_seconds + 1, size=N_ROWS)
timestamps = start_ts + pd.to_timedelta(rand_seconds, unit="s")
hours = pd.DatetimeIndex(timestamps).hour

# ─────────────────────────────────────────────
# Crime types + genders (same as v1)
# ─────────────────────────────────────────────
crime_types = ["harassment","assault","theft","stalking","eve_teasing","safe_observation"]
crime_probs = np.array([0.22, 0.08, 0.30, 0.06, 0.04, 0.30])
crime_choice = np.random.choice(crime_types, size=N_ROWS, p=crime_probs)

victim_gender = np.random.choice(["female","male","other"], size=N_ROWS, p=[0.62,0.36,0.02])
offender_gender = np.random.choice(["male","female","unknown"], size=N_ROWS, p=[0.72,0.18,0.10])

mask_hs = np.isin(crime_choice, ["harassment","stalking","eve_teasing"])
if mask_hs.sum() > 0:
    victim_gender[mask_hs] = np.random.choice(["female","male","other"], size=mask_hs.sum(), p=[0.82,0.17,0.01])

# ─────────────────────────────────────────────
# Severity (same as v1)
# ─────────────────────────────────────────────
base_severity = {
    "harassment": 0.45, "assault": 0.75, "theft": 0.30,
    "stalking": 0.60, "eve_teasing": 0.40, "safe_observation": 0.05
}
severity = np.array([base_severity[c] for c in crime_choice], dtype=float)

loc_risk = ((np.arange(n_loc) % 7) / 7.0) * 0.6
unk_risk = 0.25
loc_risk_map = {loc_df.loc[i, "name"]: float(loc_risk[i]) for i in range(n_loc)}

severity = severity + np.array([loc_risk_map.get(locality_assigned[i], unk_risk) for i in range(N_ROWS)])
severity = np.clip(severity + np.random.normal(0, 0.12, size=N_ROWS), 0, 1)

# ─────────────────────────────────────────────
# Lighting, crowd, POI (same as v1)
# ─────────────────────────────────────────────
base_lighting_per_loc = {
    loc_df.loc[i, "name"]: float(np.clip(6.0 + np.random.normal(0, 1.2), 0, 10))
    for i in range(n_loc)
}
base_crowd_per_loc = {
    loc_df.loc[i, "name"]: float(np.clip(0.45 + np.random.normal(0, 0.12), 0, 1))
    for i in range(n_loc)
}

lighting = np.empty(N_ROWS, dtype=float)
crowd = np.empty(N_ROWS, dtype=float)
poi = np.empty(N_ROWS, dtype=int)

for i in range(N_ROWS):
    loc_name = locality_assigned[i]
    if loc_name == "UNK":
        base_light = 5.0 + np.random.normal(0, 1.3)
        base_crow = 0.45 + np.random.normal(0, 0.12)
        base_poi = np.random.poisson(3)
    else:
        base_light = base_lighting_per_loc.get(loc_name, 5.5)
        base_crow = base_crowd_per_loc.get(loc_name, 0.45)
        base_poi = max(0, int(np.random.poisson(5)))

    hour = hours[i]
    if 6 <= hour <= 18:
        diurnal = 0.7
    elif 19 <= hour <= 23:
        diurnal = 1.0
    else:
        diurnal = 0.45

    lighting[i] = np.clip(base_light * diurnal + np.random.normal(0, 0.6), 0, 10)
    crowd[i] = np.clip(base_crow + (0.12 if (6 <= hour <= 9 or 17 <= hour <= 22) else -0.03) + np.random.normal(0, 0.08), 0, 1)
    poi[i] = base_poi

# ─────────────────────────────────────────────
# NEW: Lux reading + expected lux + shadow penalty
# ─────────────────────────────────────────────
is_night = ((hours >= 20) | (hours < 6)).astype(int)
is_daytime = 1 - is_night

# Generate expected_night_lux and lux_reading per incident
expected_night_lux = np.empty(N_ROWS, dtype=float)
lux_reading = np.empty(N_ROWS, dtype=float)

for i in range(N_ROWS):
    lt = location_type_assigned[i]
    profile = LOCATION_TYPE_PROFILES.get(lt, LOCATION_TYPE_PROFILES["residential"])
    lo, hi = profile["expected_night_lux_range"]

    # Expected lux for this location type
    expected_night_lux[i] = np.random.uniform(lo, hi)

    hour = hours[i]
    if 6 <= hour <= 18:
        # Daytime: lux is mostly high (100-1000), occasionally low (phone in pocket)
        if np.random.rand() < 0.05:  # 5% chance phone is in pocket
            lux_reading[i] = np.random.uniform(0, 5)
        else:
            lux_reading[i] = np.random.uniform(100, 1000)
    else:
        # Nighttime: lux depends on location type + randomness
        # Most readings cluster around expected, some anomalously low
        base_lux = expected_night_lux[i]
        if np.random.rand() < 0.15:  # 15% chance of anomalously dark
            lux_reading[i] = max(0, np.random.uniform(0, profile["alert_threshold"]))
        else:
            lux_reading[i] = max(0, base_lux + np.random.normal(0, base_lux * 0.3))

lux_reading = np.round(lux_reading, 1)
expected_night_lux = np.round(expected_night_lux, 1)

# Lux anomaly: how much darker than expected (0 if at or above expected)
# Only meaningful at night; daytime anomaly = 0
lux_anomaly = np.zeros(N_ROWS, dtype=float)
night_mask = is_night == 1
lux_anomaly[night_mask] = np.clip(
    (expected_night_lux[night_mask] - lux_reading[night_mask]) / (expected_night_lux[night_mask] + 1e-6),
    0, 1
)
lux_anomaly = np.round(lux_anomaly, 4)

# Shadow penalty: location is commercial/safe_ip type but unusually dark
shadow_penalty = np.zeros(N_ROWS, dtype=float)
for i in range(N_ROWS):
    if is_night[i] == 0:
        continue  # Temporal coupling: no shadow penalty during daytime
    lt = location_type_assigned[i]
    profile = LOCATION_TYPE_PROFILES.get(lt, LOCATION_TYPE_PROFILES["residential"])
    if lux_reading[i] < profile["shadow_lux_threshold"]:
        shadow_penalty[i] = profile["shadow_penalty_value"]

shadow_penalty = np.round(shadow_penalty, 4)

# ─────────────────────────────────────────────
# Engineered features (v1 + new)
# ─────────────────────────────────────────────
light_crowd = lighting * crowd
severity_lowlight = severity * (10.0 - lighting)
poi_to_crowd = poi / (crowd + 1e-6)
lux_x_night = lux_anomaly * is_night   # interaction: lux anomaly only matters at night

# ─────────────────────────────────────────────
# Policing + safety calculation (updated with shadow penalty)
# ─────────────────────────────────────────────
policing_per_loc = {
    loc_df.loc[i, "name"]: float(np.clip(0.5 + 0.5 * np.sin(i * 1.23), 0.05, 0.95))
    for i in range(n_loc)
}
policing_array = np.array([policing_per_loc.get(locality_assigned[i], 0.5) for i in range(N_ROWS)])

w_sev = 0.55 + 0.15 * (1 - policing_array)
w_light = 0.30 + 0.08 * (1 - policing_array)
w_crowd = 0.15 - 0.03 * (1 - policing_array)
w_sum = w_sev + w_light + w_crowd
w_sev /= w_sum; w_light /= w_sum; w_crowd /= w_sum

base_safety = 1.0 - (w_sev * severity + w_light * (1.0 - lighting / 10.0) + w_crowd * (1.0 - crowd))

# Apply shadow penalty to base safety (reduces score when location is dark+commercial)
base_safety = base_safety * (1.0 - shadow_penalty)

sigma = 0.02 + 0.08 * severity * (1.0 - crowd)
label_noise = np.random.normal(0, sigma, size=N_ROWS)

flip_mask = (np.random.rand(N_ROWS) < 0.004)
label_shift_vals = np.random.uniform(-0.18, -0.05, size=flip_mask.sum())
label_shift_full = np.zeros(N_ROWS, dtype=float)
label_shift_full[flip_mask] = label_shift_vals

safety_index = np.clip(base_safety + label_noise + label_shift_full, 0.0, 1.0)

# ─────────────────────────────────────────────
# Assemble DataFrame
# ─────────────────────────────────────────────
df = pd.DataFrame({
    "incident_id": np.arange(1, N_ROWS + 1),
    "timestamp": timestamps,
    "latitude": lat,
    "longitude": lon,
    "locality": locality_assigned,
    "location_type": location_type_assigned,       # NEW
    "crime_type": crime_choice,
    "victim_gender": victim_gender,
    "offender_gender": offender_gender,
    "severity_score": np.round(severity, 4),
    "lighting_score": np.round(lighting, 3),
    "crowd_density": np.round(crowd, 3),
    "poi_density": poi,
    "lux_reading": lux_reading,                     # NEW
    "expected_night_lux": expected_night_lux,        # NEW
    "lux_anomaly": lux_anomaly,                      # NEW
    "shadow_penalty": shadow_penalty,                # NEW
    "light_crowd": np.round(light_crowd, 3),
    "severity_lowlight": np.round(severity_lowlight, 3),
    "poi_to_crowd": np.round(poi_to_crowd, 3),
    "lux_x_night": np.round(lux_x_night, 4),        # NEW
    "policing_index": np.round(policing_array, 3),
    "safety_index": np.round(safety_index, 4),
    "hour_of_day": hours,
    "is_weekend": (pd.DatetimeIndex(timestamps).dayofweek >= 5).astype(int),
    "is_night": is_night,
    "is_daytime": is_daytime,                        # NEW
})

# ─────────────────────────────────────────────
# Save feature_order.json
# ─────────────────────────────────────────────
FEATURE_ORDER = [c for c in df.columns if c != "safety_index"]
Path(OUT_FEATURE_ORDER).write_text(json.dumps(FEATURE_ORDER), encoding="utf-8")
print(f"Saved feature order to {OUT_FEATURE_ORDER} ({len(FEATURE_ORDER)} features)")

# ─────────────────────────────────────────────
# locality_aggregates.json (v2: includes location_type + expected_night_lux)
# ─────────────────────────────────────────────
# Build a map of location_type per locality from the CSV
loc_type_map = dict(zip(loc_df["name"], loc_df["location_type"]))

# Expected night lux midpoint per type
expected_lux_midpoint = {}
for lt, prof in LOCATION_TYPE_PROFILES.items():
    lo, hi = prof["expected_night_lux_range"]
    expected_lux_midpoint[lt] = (lo + hi) / 2

loc_agg = df.groupby("locality").agg(
    mean_safety=("safety_index", "mean"),
    median_safety=("safety_index", "median"),
    n_incidents=("safety_index", "count"),
    mean_severity=("severity_score", "mean"),
    median_lighting=("lighting_score", "median"),
    median_crowd=("crowd_density", "median"),
    mean_lux_anomaly=("lux_anomaly", "mean"),
    mean_shadow_penalty=("shadow_penalty", "mean"),
).reset_index()

loc_dict = {}
for _, row in loc_agg.iterrows():
    loc_name = row["locality"]
    lt = loc_type_map.get(loc_name, "residential")
    loc_dict[loc_name] = {
        "mean_safety": round(float(row["mean_safety"]), 4),
        "median_safety": round(float(row["median_safety"]), 4),
        "n_incidents": int(row["n_incidents"]),
        "mean_severity": round(float(row["mean_severity"]), 4),
        "median_lighting": round(float(row["median_lighting"]), 4),
        "median_crowd": round(float(row["median_crowd"]), 4),
        "location_type": lt,
        "expected_night_lux": expected_lux_midpoint.get(lt, 15),
        "mean_lux_anomaly": round(float(row["mean_lux_anomaly"]), 4),
        "mean_shadow_penalty": round(float(row["mean_shadow_penalty"]), 4),
    }

with open(OUT_LOCALITY_AGG, "w", encoding="utf-8") as f:
    json.dump(loc_dict, f, indent=2)

print(f"Saved locality aggregates to: {OUT_LOCALITY_AGG}  (localities: {len(loc_dict)})")

# Save incidents CSV
df.to_csv(OUT_INCIDENTS, index=False)
print(f"Saved synthetic incidents to: {OUT_INCIDENTS}  (rows: {len(df)})")
print(f"\nNew columns: location_type, lux_reading, expected_night_lux, lux_anomaly, shadow_penalty, lux_x_night, is_daytime")
print("All done.")